In [0]:
# Silver layer: suppliers (B2B supply-side companies)
# Source: b2b_ecommerce.bronze.suppliers (append-only, connector-managed
#         and already typed: int ids, int lead_time_days / min_order_value,
#         boolean is_preferred, timestamps)
# Grain:  one row per supplier_id (SCD Type 1 - latest updated_at wins,
#         no history). company_id is unique
#         1:1 with silver.companies; the FK is enforced against current
#         silver rows of company_type='supplier' (verified: all 25
#         referenced companies are type 'supplier').
# Approach: bronze.suppliers is already clean (0 nulls in every column,
#           0 padding, 0 invalid emails, 0 orphans vs bronze.companies),
#           so we only trim/lower the contact text and keep the typed
#           columns as-is. AUTO CDC keyed by supplier_id MERGEs into
#           b2b_ecommerce.silver.suppliers, which deduplicates identical
#           re-sends and keeps re-ingestion idempotent. SCD Type 1 (user
#           decision 2026-11-18): a changed supplier re-synced into bronze
#           overwrites the existing row in place - only the latest state
#           is kept, no history.
from pyspark import pipelines as dp
from pyspark.sql import functions as F

BRONZE_SUPPLIERS = "b2b_ecommerce.bronze.suppliers"
# current companies live in the silver layer (same pipeline dataset)
SILVER_COMPANIES = "companies"


def clean_text(name, lower=False):
    """Trim a text column; a blank value ("" or " ") becomes NULL, not ""."""
    trimmed = F.trim(F.col(name))
    if lower:
        trimmed = F.lower(trimmed)
    return (
        F.when(trimmed != "", trimmed)
        .otherwise(F.lit(None).cast("string"))
        .alias(name)
    )


# ---------------------------------------------------------------------------
# 1) Cleaned stream of bronze.suppliers (+ FK match flag vs current
#    silver.companies rows of type 'supplier')
# ---------------------------------------------------------------------------
@dp.temporary_view(name="suppliers_clean_stream")
# Data quality validation (only checks that can actually catch bad rows).
# Drop-level: without the merge key or the CDC sequence the row cannot be
# merged at all - quarantine it out of silver; the raw value stays in
# bronze for inspection.
@dp.expect_or_drop("supplier_id_not_null", "supplier_id IS NOT NULL")
@dp.expect_or_drop("updated_at_not_null", "updated_at IS NOT NULL")
# Referential integrity: every supplier must be a CURRENT silver company of
# type 'supplier' - a missing company or a link to a buyer fails the whole
# update instead of silently landing in silver (hard stop is intended, same
# policy as silver.customers; this table is a small dimension, so an orphan
# means the source is broken and someone must intervene). No separate
# company_id_not_null warn check: a NULL company_id never matches the join,
# so company_id_exists already catches it - that warn check was dead.
@dp.expect_or_fail("company_id_exists", "_company_id_match IS NOT NULL")
# Warn-level: the row is kept and the problem only recorded in the event
# log (same policy as silver.customers email/phone checks).
@dp.expect("contact_name_not_null", "contact_name IS NOT NULL")
@dp.expect("contact_email_valid", "contact_email IS NULL OR contact_email RLIKE '^[^@ ]+@[^@ ]+[.][^@ ]+$'")
@dp.expect("lead_time_days_not_negative", "lead_time_days IS NULL OR lead_time_days >= 0")
@dp.expect("min_order_value_not_negative", "min_order_value IS NULL OR min_order_value >= 0")
def suppliers_clean_stream():
    # bronze.suppliers is append-only: a changed supplier arrives as a new
    # row. The AUTO CDC merge below turns it into an in-place update of the
    # same supplier_id (Type 1: the row with the latest updated_at wins),
    # so identical re-sends collapse and re-ingestion is idempotent.
    bronze = spark.readStream.table(BRONZE_SUPPLIERS)

    # stream-static LEFT JOIN against CURRENT silver.companies rows of type
    # 'supplier' only: a company that exists in bronze but was dropped from
    # silver, or a buyer company, does not match - the row becomes an orphan
    # and the fail-level expectation stops the update. silver.companies is an
    # upsert (SCD Type 1): one row per company_id, no version columns - every
    # row is current. An expectation is a per-row column expression, so the
    # FK check needs the match flag as a column.
    companies = (
        spark.read.table(SILVER_COMPANIES)
        .filter(F.col("company_type") == "supplier")
        .select(F.col("company_id").alias("_company_id_match"))
        .distinct()
    )

    return bronze.join(
        companies,
        bronze["company_id"] == companies["_company_id_match"],
        "left",
    ).select(
        F.col("supplier_id").cast("int").alias("supplier_id"),
        F.col("company_id").cast("int").alias("company_id"),
        F.col("lead_time_days").cast("int").alias("lead_time_days"),
        clean_text("contact_name"),
        # canonical lowercase email (same form as silver.customers)
        clean_text("contact_email", lower=True),
        F.col("min_order_value").cast("int").alias("min_order_value"),
        F.col("is_preferred").cast("boolean").alias("is_preferred"),
        F.col("created_at").cast("timestamp").alias("created_at"),
        F.col("updated_at").cast("timestamp").alias("updated_at"),
        F.col("_company_id_match"),  # internal FK flag, never lands in silver
    )


# ---------------------------------------------------------------------------
# 2) Silver target: b2b_ecommerce.silver.suppliers
#    Rows are MERGEd by supplier_id (SCD Type 1): the row with the latest
#    updated_at wins - updates overwrite in place, no history, identical
#    re-sends are no-ops.
# ---------------------------------------------------------------------------
dp.create_streaming_table(
    name="suppliers",
    comment=(
        "Silver suppliers: cleaned and standardized; one row per supplier_id "
        "(SCD Type 1 merge - latest updated_at wins, no history)"
    ),
)

dp.create_auto_cdc_flow(
    target="suppliers",
    source="suppliers_clean_stream",
    keys=["supplier_id"],
    sequence_by="updated_at",
    stored_as_scd_type=1,
    # internal FK flag must not land in the target table
    except_column_list=["_company_id_match"],
)